In [26]:
import json
import os
import re
from pathlib import Path

import fiona
import geopandas as gpd
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURAÇÃO DE PATHS (Adaptado para Jupyter Notebook)
# ==============================================================================
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path(os.getcwd())
    # Se o notebook estiver dentro de 'notebooks/', sobe um nível automaticamente
    if BASE_DIR.name in ["notebooks", "scripts"]:
        BASE_DIR = BASE_DIR.parent

GPKG        = BASE_DIR / "data" / "dados_insa.gpkg"
STYLES_JSON = BASE_DIR / "src" / "assets" / "styles.json"
OUTPUT_JSON = BASE_DIR / "src" / "assets" / "stats.json"

TARGET_CRS       = "EPSG:5880"  # Projeção Albers Brasil (Métrica)
MATCH_THRESHOLD  = 0.80
FLOAT_TOL        = 1e-6

# ==============================================================================
# FUNÇÕES AUXILIARES INDEPENDENTES
# ==============================================================================
def is_stroke_only(layer_styles):
    return all(v.startswith("stroke:") for v in layer_styles.values())


def _parse_floats(keys):
    result = {}
    for k in keys:
        try:
            result[k] = float(k)
        except (ValueError, TypeError):
            pass
    return result


def find_exact_field(gdf, expected_keys):
    n = len(expected_keys)
    if n == 0:
        return None, 0.0, 0

    expected_floats = _parse_floats(expected_keys)
    all_numeric = len(expected_floats) == n
    best_col, best_score, best_n_unique = None, 0.0, 0

    for col in gdf.columns:
        if col == "geometry":
            continue

        unique_vals = gdf[col].dropna().unique()
        n_unique = len(unique_vals)

        unique_str = {str(v) for v in unique_vals}
        score = len(expected_keys & unique_str) / n
        if score > best_score:
            best_col, best_score, best_n_unique = col, score, n_unique
        if best_score >= 1.0:
            break

        if all_numeric and score < MATCH_THRESHOLD:
            try:
                unique_floats = {float(v) for v in unique_vals}
                matched = sum(
                    1 for ef in expected_floats.values()
                    if any(abs(ef - uf) < FLOAT_TOL for uf in unique_floats)
                )
                score_f = matched / n
                if score_f > best_score:
                    best_col, best_score, best_n_unique = col, score_f, n_unique
            except (ValueError, TypeError):
                pass

    if best_score >= MATCH_THRESHOLD:
        return best_col, best_score, best_n_unique
    return None, best_score, 0


def find_ranged_field(gdf, sorted_boundaries):
    lo, hi = sorted_boundaries[0], sorted_boundaries[-1]
    for col in gdf.columns:
        if col == "geometry":
            continue
        series = gdf[col].dropna()
        if not pd.api.types.is_numeric_dtype(series) or len(series) == 0:
            continue
        col_min, col_max = float(series.min()), float(series.max())
        if col_min <= hi and col_max >= lo:
            return col
    return None


def assign_range_class(value, sorted_boundaries):
    try:
        v = float(value)
    except (ValueError, TypeError):
        return None
    for b in sorted_boundaries:
        if v <= b + FLOAT_TOL:
            return b
    return sorted_boundaries[-1]


def parse_label_bounds(keys):
    result = {}
    for k in keys:
        nums = re.findall(r'-?\d+\.?\d*', k)
        if not nums:
            return None
        result[k] = float('inf') if '>' in k else float(nums[-1])
    return result if len(result) == len(keys) else None


def color_for_exact_value(val, layer_styles, expected_floats):
    str_val = str(val)
    if str_val in layer_styles:
        return layer_styles[str_val]
    if expected_floats:
        try:
            val_f = float(val)
            for k, kf in expected_floats.items():
                if abs(val_f - kf) < FLOAT_TOL:
                    return layer_styles[k]
        except (ValueError, TypeError):
            pass
    return None

# ==============================================================================
# FUNÇÃO PRINCIPAL DE PROCESSAMENTO E DISSOLVE
# ==============================================================================
def compute_stats(layer_name, layer_styles):
    expected_keys = set(layer_styles.keys())
    expected_floats = _parse_floats(expected_keys)
    all_numeric = len(expected_floats) == len(expected_keys)

    # 1. Carrega a camada do GeoPackage
    gdf = gpd.read_file(GPKG, layer=layer_name)

    # 2. Descobre o campo correspondente por similaridade de chaves
    field, _, n_unique = find_exact_field(gdf, expected_keys)
    if field is not None and all_numeric and n_unique > len(expected_keys) * 2:
        field = None

    # 3. Força a reprojeção antecipada para cálculo métrico de área exato
    if gdf.crs != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)

    # --- CASO A: CORRESPONDÊNCIA EXATA DE CATEGORIAS/VALORES DISCRETOS ---
    if field is not None:
        # Arredonda se for coluna numérica usando validação segura do Pandas
        if pd.api.types.is_numeric_dtype(gdf[field]):
            gdf[field] = gdf[field].round(4)
            
        # Limpa strings e converte dados
        gdf[field] = gdf[field].fillna('Nao Informado').astype(str).str.strip()
        
        # Mapeia floats imprecisos da tabela de atributos para strings exatas do JSON
        if all_numeric:
            def normalizar_chave(val):
                try:
                    vf = float(val)
                    for k, kf in expected_floats.items():
                        if abs(vf - kf) < FLOAT_TOL:
                            return k
                except (ValueError, TypeError):
                    pass
                return val
            gdf[field] = gdf[field].apply(normalizar_chave)

        # EXECUTA O DISSOLVE GEOMÉTRICO NATIVO DO GEOPANDAS
        gdf_dissolved = gdf.dissolve(by=field, as_index=False)
        gdf_dissolved["_area_km2"] = gdf_dissolved.geometry.area / 1e6

        classes = []
        for _, row in gdf_dissolved.iterrows():
            val = row[field]
            area = row["_area_km2"]
            color = color_for_exact_value(val, layer_styles, expected_floats)
            if color is None:
                continue
            classes.append({
                "label": str(val),
                "area_km2": round(float(area), 1),
                "color": color,
            })

        classes.sort(key=lambda x: x["area_km2"], reverse=True)
        total_km2 = round(sum(c["area_km2"] for c in classes), 1)
        return {"classes": classes, "total_km2": total_km2, "field_used": field}, None

    # --- CASO B: PROCESSAMENTO DE RANGES / INTERVALOS DE CLASSES ---
    if all_numeric:
        bounds_map = {v: k for k, v in expected_floats.items()}
        sorted_bounds = sorted(expected_floats.values())
    else:
        label_bounds = parse_label_bounds(expected_keys)
        if label_bounds is None:
            return None, "exact match falhou e chaves não têm intervalos reconhecíveis"
        bounds_map = {v: k for k, v in label_bounds.items()}
        sorted_bounds = sorted(lb for lb in label_bounds.values() if lb != float('inf'))
        if label_bounds and max(label_bounds.values()) == float('inf'):
            sorted_bounds.append(float('inf'))

    finite_bounds = [b for b in sorted_bounds if b != float('inf')]
    field = find_ranged_field(gdf, finite_bounds if finite_bounds else sorted_bounds)
    if field is None:
        return None, "nenhum campo numérico com intervalo compatível encontrado"

    # Converte valores contínuos para os intervalos esperados
    gdf["_class"] = gdf[field].apply(
        lambda v: assign_range_class(v, sorted_bounds) if pd.notna(v) else None
    )
    gdf = gdf.dropna(subset=["_class"])
    gdf["_class"] = gdf["_class"].apply(lambda b: bounds_map[b])

    # EXECUTA O DISSOLVE GEOMÉTRICO BASEADO NOS INTERVALOS MAPEADOS
    gdf_dissolved = gdf.dissolve(by="_class", as_index=False)
    gdf_dissolved["_area_km2"] = gdf_dissolved.geometry.area / 1e6

    classes = []
    for _, row in gdf_dissolved.iterrows():
        key = row["_class"]
        area = row["_area_km2"]
        color = layer_styles[key]
        classes.append({
            "label": key,
            "area_km2": round(float(area), 1),
            "color": color,
        })

    classes.sort(key=lambda x: x["area_km2"], reverse=True)
    total_km2 = round(sum(c["area_km2"] for c in classes), 1)
    return {
        "classes": classes,
        "total_km2": total_km2,
        "field_used": field,
    }, None

# ==============================================================================
# EXECUÇÃO DO PIPELINE
# ==============================================================================
def main():
    print(f"Diretório Base reconhecido: {BASE_DIR}")
    
    if not STYLES_JSON.exists():
        print(f"❌ Erro: O arquivo de estilos não foi encontrado em: {STYLES_JSON}")
        return
        
    with open(STYLES_JSON, encoding="utf-8") as f:
        styles = json.load(f)

    available_layers = set(fiona.listlayers(str(GPKG)))

    result = {}
    processed = 0
    skipped = []

    for layer_name, layer_styles in styles.items():
        if is_stroke_only(layer_styles):
            result[layer_name] = None
            skipped.append(f"{layer_name} — stroke-only")
            continue

        if layer_name not in available_layers:
            result[layer_name] = None
            skipped.append(f"{layer_name} — não encontrado no GeoPackage")
            continue

        data, err = compute_stats(layer_name, layer_styles)
        if err:
            result[layer_name] = None
            skipped.append(f"{layer_name} — {err}")
            print(f"⚠️  Pulando {layer_name}: {err}")
        else:
            result[layer_name] = data
            processed += 1
            n = len(data["classes"])
            print(f"✓  {layer_name} (campo identificado: {data['field_used']}, {n} classes consolidadas)")

    print("\n--- Relatório de Camadas Ignoradas ---")
    for s in skipped:
        print(f"   Pulando {s}")

    # Salva o arquivo final de estatísticas
    os.makedirs(OUTPUT_JSON.parent, exist_ok=True)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    size_kb = os.path.getsize(OUTPUT_JSON) // 1024
    print(f"\nConcluído! {processed} camadas processadas com sucesso.")
    print(f"Arquivo JSON gerado em: {OUTPUT_JSON} ({size_kb} KB)")


if __name__ == "__main__":
    main()

Diretório Base reconhecido: /Users/marcellodebarrosfilho/code/insa-front
✓  iqs_sab_pb (campo identificado: IQS, 5 classes consolidadas)
✓  declividade_sab_pb_pesos (campo identificado: PesDecl, 5 classes consolidadas)
✓  geologia_sab_pb_original (campo identificado: GLO_DS_LIT, 52 classes consolidadas)
✓  geologia_sab_pb_pesos (campo identificado: pes_Peso, 5 classes consolidadas)
✓  solos_tipos_sab_pb_original (campo identificado: DSC_COMPON, 17 classes consolidadas)
✓  solos_tipos_sab_pb_pesos (campo identificado: TipSoilPes, 4 classes consolidadas)
✓  textura_sab_pb_original (campo identificado: DSC_TEXTUR, 3 classes consolidadas)
✓  textura_sab_pb_pesos (campo identificado: SoilTextur, 3 classes consolidadas)
✓  eto_sab_pb_original (campo identificado: ETo_Climat, 5 classes consolidadas)
✓  eto_sab_pb_pesos (campo identificado: ETo_Pesos, 5 classes consolidadas)
✓  ia_sab_pb_original (campo identificado: IA_climat, 2 classes consolidadas)
✓  ia_sab_pb_pesos (campo identificado: IA